# Proyecto Oráculo — Solución NPO (Llama 3.2 3B)

Análisis y Diseño de Algoritmos

Esta es una solución de ejemplo para la celda 4: implementa **Naive Prompt
Optimization (NPO)** [1], con un teacher servido por OpenRouter y el estudiante
`llama3b` (`unsloth/Llama-3.2-3B-Instruct`). El Algoritmo 1 está citado en la sección 4.

[1] Yuan Chang y Xiaoqi Chen, *Naive Prompt Optimization: Rethinking the Need for Complex Prompt Search*, arXiv:2608.27266, 2026. https://arxiv.org/abs/2608.27266

NPO no recorre el catálogo: un modelo teacher mira las trazas de fallo y propone
la siguiente config del linaje. El oráculo sigue siendo la caja negra — solo se
consulta `evaluar` / `validar`.

```
r = oraculo.evaluar(config, instancias, semilla)
r_val = oraculo.validar(config, n)   # partición de validación, muestra fija

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]
```


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

Misma instalación que el notebook del curso. El reinicio es obligatorio: si siguen sin reiniciar, `transformers` a veces queda a medias y `cargar_modelo` falla con un error opaco.


In [ ]:
# bitsandbytes: checkpoints 4-bit. nltk/spacy/emoji/langdetect: los usa el
# verificador de open-instruct, no este notebook.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo — `llama3b`

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca. Este es el estudiante; el teacher va por OpenRouter en la celda 4.


In [ ]:
from ayudas import cargar_modelo

# Estudiante de este notebook. El nombre entra en la clave de caché.
modelo = cargar_modelo("llama3b")  # unsloth/Llama-3.2-3B-Instruct


## 3 · El oráculo

La corrida de NPO es larga (varios cientos de rollouts): montamos Drive para no
perder el caché si Colab se desconecta. Cada notebook de solución usa un JSON
distinto (`cache_npo_<modelo>.json`) para no mezclar respuestas de otro estudiante.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, RANURAS, TEMPERATURAS, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
# Path en Drive: una desconexión de Colab no tira las generaciones ya pagadas.
oraculo = Oraculo(
    modelo, busqueda, validacion,
    cache="/content/drive/MyDrive/cache_npo_llama3b.json",
)

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Naive Prompt Optimization (NPO)

La celda de código implementa el **Algoritmo 1** de Chang y Chen (2026) [1]: un solo
linaje de prompts, actualizado con una ventana deslizante de trazas de rollout.

> **Algorithm 1** Naive Prompt Optimization with Sliding-Window Rollout Feedback
>
> **Require:** Initial prompt $\mathcal{P}^{(0)}$, sample task dataset $\mathcal{D}$ from environment, minibatch size $N$, window size $W$, teacher $\mathcal{T}$, iterations $Y$
>
> 1. **for** $i = 0$ **to** $Y-1$ **do**
> 2. Sample minibatch $\mathcal{B}_i$ of size $N$ from $\mathcal{D}$
> 3. Run the target model with prompt $\mathcal{P}^{(i)}$ on $\mathcal{B}_i$
> 4. Collect rollout traces and rewards $\mathcal{R}_i$
> 5. Construct sliding-window feedback: $\{ \mathcal{R}_j \}_{j=\max(0,i-W+1)}^{i}$
> 6. Generate the next prompt: $\mathcal{P}^{(i+1)} \leftarrow \mathcal{T}\!\left(\mathcal{P}^{(i)}, \{ \mathcal{R}_j \}_{j=\max(0,i-W+1)}^{i}\right)$
> 7. **end for**
> 8. **return** Prompt sequence $\{ \mathcal{P}^{(0)}, \ldots, \mathcal{P}^{(Y)} \}$ and best candidate
>
> Chang, Y. y Chen, X. (2026). *Naive Prompt Optimization: Rethinking the Need for Complex Prompt Search*. arXiv:2608.27266.

[1] Yuan Chang y Xiaoqi Chen, *Naive Prompt Optimization: Rethinking the Need for Complex Prompt Search*, arXiv:2608.27266, 2026. https://arxiv.org/abs/2608.27266

Lo que distingue a NPO de OPRO: el teacher recibe **trazas completas y recompensa por
rollout**, no solo prompts previos y un escalar. OPRO solo ve “este prompt sacó 0.4”;
NPO ve *por qué* falló cada rollout (restricción violada + texto producido).

En este proyecto el prompt no es texto libre: se arma escogiendo un índice por ranura
del `CATALOGO`. Por eso el teacher, en cada iteración, no reescribe texto sino que
propone la siguiente config del linaje — el resto del algoritmo (linaje único, minibatch
fresco, ventana deslizante, selección final en validación) se mantiene igual.

Hiperparámetros: los del paper para IFBench — `N=50`, `Y=10`, `N_VAL=300`. Este split
visible tiene 300 instancias de validación y `validar` las usa todas. El presupuesto de
3.500 rollouts del paper sale de 50×10 (búsqueda) + 300×10 (validación): NPO valida
**cada versión** del linaje, no solo la última, y de ahí sale el mejor candidato.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  NPO — Naive Prompt Optimization
#  Chang y Chen (2026), arXiv:2608.27266, Algorithm 1
#  Linaje único + ventana deslizante de trazas. Teacher: OpenRouter.
#  Estudiante: el modelo cargado arriba, a través del oráculo.
# ═══════════════════════════════════════════════════════════════════════

import getpass
import json
import logging
import random
import re

import requests

logging.getLogger("absl").setLevel(logging.CRITICAL)  # ruido de langdetect en salidas sin palabras

# ─── Teacher ───────────────────────────────────────────────────────────
API_KEY = getpass.getpass("OpenRouter API key: ")
TEACHER = "openai/gpt-5.1"   # verifiquen el id en openrouter.ai/models


def teacher(mensaje):
    """Llama al modelo teacher (OpenRouter) y devuelve el texto de la respuesta.

    Temperatura 1.0: el paper quiere propuestas diversas, no el modo greedy.
    Si el JSON sale malformado, `_leer_config` descarta y deja la config anterior.
    """
    r = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": TEACHER,
            "messages": [{"role": "user", "content": mensaje}],
            "temperature": 1.0,
        },
        timeout=180,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


# ─── Hiperparámetros del paper (IFBench) ───────────────────────────────
N = 50            # tamaño del minibatch B_i
W = 3             # ventana deslizante (el paper la ajusta al contexto del teacher)
Y = 10            # iteraciones del linaje
N_VAL = 300       # validación: este split tiene 300; validar() las toma todas
SEMILLA = 1
MAX_TRAZAS = 8    # fallos que se le muestran al teacher por iteración (el contexto no da para más)
LARGO = 400       # chars de cada salida del estudiante en el mensaje al teacher


# ─── Armado del mensaje al teacher ─────────────────────────────────────
META = """Eres el modelo TEACHER de Naive Prompt Optimization (NPO).

Un modelo ESTUDIANTE resuelve tareas de seguimiento de instrucciones (estilo IFEval):
recibe una petición en inglés con restricciones verificables (número de palabras, formato,
frase final, mayúsculas, etc.). Un verificador automático da reward 1 si las cumple TODAS
y 0 si viola alguna.

El prompt del estudiante no es texto libre: se arma escogiendo un índice por ranura de este
catálogo fijo, y cada texto elegido se pega DESPUÉS de la petición: primero las opciones seguras (último índice de cada ranura), después las dañinas.

CATÁLOGO
{catalogo}

Estas son las últimas {w} iteraciones de la optimización, cada una con su config, su
recompensa media y las trazas de los rollouts que fallaron:

{ventana}

Configs ya probadas (evita repetirlas sin una buena razón):
{probadas}

Tu trabajo: proponer la SIGUIENTE config del linaje, razonando sobre POR QUÉ fallaron esos
rollouts concretos (mira la restricción violada y el texto que produjo el estudiante).
Responde SOLO con un objeto JSON, sin markdown ni texto alrededor:
{{"razon": "<una frase>", "rol": <int>, "estrategia": <int>, "formato": <int>,
  "estilo": <int>, "verificacion": <int>, "recordatorio": <int>, "temperatura": <float>}}"""


def _catalogo_txt():
    """Catálogo numerado para el teacher: sin esto no sabe qué significa rol=2."""
    lineas = []
    for ranura in RANURAS:
        lineas.append(f"{ranura}:")
        for i, texto in enumerate(CATALOGO[ranura]):
            lineas.append(f"  {i} = {texto!r}" if texto else f"  {i} = (vacío)")
    lineas.append("temperatura: " + ", ".join(str(t) for t in TEMPERATURAS))
    return "\n".join(lineas)


def _feedback(paso):
    """Una iteración de la ventana: config + recompensa + trazas de rollout.

    Recorta a MAX_TRAZAS y LARGO: el teacher no necesita 50 salidas completas
    para ver el patrón de fallo, y el contexto se llena rápido.
    """
    p = [
        f"### Iteración {paso['i']}",
        f"config: {json.dumps(paso['config'])}",
        f"recompensa media: {paso['precision']:.3f}"
        f"  ({paso['n'] - len(paso['trazas'])}/{paso['n']} rollouts con reward 1)",
    ]
    if paso["trazas"]:
        p.append("rollouts con reward 0:")
        for t in paso["trazas"][:MAX_TRAZAS]:
            p.append(f"- restricción violada: {t['violo']}")
            p.append(f"  salida del estudiante: {t['salida'][:LARGO]!r}")
    else:
        p.append("ningún fallo en este minibatch.")
    return "\n".join(p)


def _leer_config(texto, anterior):
    """JSON del teacher → config válida. Ante cualquier duda, deja la anterior.

    Índices fuera de rango o temperaturas que no están en TEMPERATURAS se
    descartan ranura por ranura: un JSON a medias no tira el linaje entero.
    """
    m = re.search(r"\{.*\}", texto, re.S)
    if not m:
        return anterior, "(el teacher no devolvió JSON)"
    try:
        d = json.loads(m.group())
    except json.JSONDecodeError:
        return anterior, "(JSON inválido)"

    nueva = {}
    for ranura in RANURAS:
        i = d.get(ranura, anterior.get(ranura, 0))
        nueva[ranura] = (
            i if isinstance(i, int) and 0 <= i < len(CATALOGO[ranura]) else anterior.get(ranura, 0)
        )
    t = d.get("temperatura", anterior["temperatura"])
    nueva["temperatura"] = t if t in TEMPERATURAS else anterior["temperatura"]
    return nueva, str(d.get("razon", ""))


# ─── Algoritmo 1 ───────────────────────────────────────────────────────
# Arranca en los índices 0 (las opciones dañinas, temp 0.0): el teacher
# tiene que subir desde ahí, no desde una semilla buena.
config = {"rol": 0, "estrategia": 0, "formato": 0, "estilo": 0, "verificacion": 0, "recordatorio": 0, "temperatura": 0.0}

historia, historial, probadas = [], [], []

for i in range(Y):
    lote = random.Random(SEMILLA + i).sample(busqueda, N)      # minibatch B_i fresco
    r = oraculo.evaluar(config, lote, semilla=SEMILLA)         # rollouts + recompensas R_i

    historia.append(
        {"i": i, "config": config, "precision": r.precision, "trazas": r.trazas, "n": r.n}
    )
    historial.append(r.precision)
    probadas.append(dict(config))
    print(f"[{i}] recompensa {r.precision:5.1%}   {config}")

    if i == Y - 1:
        break

    ventana = "\n\n".join(_feedback(p) for p in historia[-W:])  # {R_j}, j = i-W+1 .. i
    mensaje = META.format(
        catalogo=_catalogo_txt(),
        w=min(W, len(historia)),
        ventana=ventana,
        probadas=json.dumps(probadas, ensure_ascii=False),
    )
    try:
        config, razon = _leer_config(teacher(mensaje), config)  # P(i+1) ← T(P(i), {R_j})
        print(f"     teacher: {razon}")
    except Exception as e:
        print(f"     teacher falló ({e}) — se repite la config")


# ─── Mejor candidato: cada versión del linaje, medida en validación ────
#  Presupuesto del paper en IFBench: 50*10 (búsqueda) + 300*10 (validación) = 3.500.
#  No se elige por el minibatch: un B_i fácil infla la recompensa de búsqueda.
mejor = None
for paso in historia:
    print(f"\n[{paso['i']}] {paso['config']}")
    rv = oraculo.validar(paso["config"], n=N_VAL)
    paso["validacion"] = rv.precision
    if mejor is None or rv.precision > mejor[0]:
        mejor = (rv.precision, paso["config"])

print(f"\nmejor config: {mejor[1]}   validación {mejor[0]:.1%}")
print(f"presupuesto: {N * Y} rollouts de búsqueda + {Y} validaciones")


### Leer los fallos de la mejor config

Cada resultado trae sus trazas: mírenlas todas las veces que quieran. `violo` es la
primera restricción que no pasó; `salida` es lo que escribió el estudiante. Sirve para
chequear a mano si el teacher estaba mirando fallos reales o ruido del minibatch.


In [ ]:
# Reusa el caché si esas 50 de búsqueda ya se midieron en el linaje.
r = oraculo.evaluar(mejor[1], busqueda[:50], semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Curva del linaje

Recompensa de minibatch por iteración vs. la validación de cada versión — así se ve
si el linaje mejora de verdad o si el minibatch solo fue más fácil esa vez.
`curva` dibuja el mejor minibatch acumulado; la tabla de abajo compara los dos números
lado a lado.


In [ ]:
from ayudas import curva

curva(historial)  # mejor recompensa de minibatch, acumulada — no la de validación

for paso in historia:
    print(f"[{paso['i']}]  minibatch {paso['precision']:5.1%}   validación {paso['validacion']:5.1%}")


## 5 · La entrega

Un `entrega.json` con grupo, config ganadora y semana. La nota no sale de este
notebook: el profesor corre esa config sobre el test privado. Cambien `G07` y
`semana` antes de descargar.


In [ ]:
from ayudas import entrega
from google.colab import files

# grupo: identificador del equipo. semana: número de semana del curso.
entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
